In [1]:
import random
from dataclasses import dataclass

# -----------------------------
# 1) A tiny GridWorld environment (from scratch)
# -----------------------------
@dataclass
class GridWorld:
    rows: int = 5
    cols: int = 5
    start: tuple = (0, 0)
    goal: tuple = (4, 4)
    pit: tuple = (3, 3)
    step_cost: float = -0.01
    max_steps: int = 200

    # Actions: 0=UP, 1=RIGHT, 2=DOWN, 3=LEFT
    actions = (0, 1, 2, 3)

    def __post_init__(self):
        self.reset()

    def reset(self):
        self.pos = self.start
        self.t = 0
        return self._state(self.pos)

    def n_states(self):
        return self.rows * self.cols

    def n_actions(self):
        return len(self.actions)

    def _state(self, pos):
        r, c = pos
        return r * self.cols + c

    def _pos_from_state(self, s):
        return (s // self.cols, s % self.cols)

    def step(self, action):
        """Return: next_state, reward, done, info"""
        r, c = self.pos
        self.t += 1

        if action == 0:   # UP
            r = max(0, r - 1)
        elif action == 1: # RIGHT
            c = min(self.cols - 1, c + 1)
        elif action == 2: # DOWN
            r = min(self.rows - 1, r + 1)
        elif action == 3: # LEFT
            c = max(0, c - 1)
        else:
            raise ValueError("Invalid action")

        self.pos = (r, c)

        # Reward logic
        done = False
        reward = self.step_cost

        if self.pos == self.goal:
            reward = 1.0
            done = True
        elif self.pos == self.pit:
            reward = -1.0
            done = True
        elif self.t >= self.max_steps:
            done = True

        return self._state(self.pos), reward, done, {}

# -----------------------------
# 2) Q-learning (tabular) from scratch
# -----------------------------
def epsilon_greedy(Q, s, epsilon):
    if random.random() < epsilon:
        return random.randrange(len(Q[s]))
    # tie-break randomly among max actions
    max_q = max(Q[s])
    best_actions = [a for a, q in enumerate(Q[s]) if q == max_q]
    return random.choice(best_actions)

def train_q_learning(
    env,
    episodes=5000,
    alpha=0.1,        # learning rate
    gamma=0.99,       # discount factor
    eps_start=1.0,
    eps_end=0.05,
    eps_decay=0.999,  # multiply epsilon each episode
):
    Q = [[0.0 for _ in range(env.n_actions())] for _ in range(env.n_states())]
    epsilon = eps_start

    for ep in range(episodes):
        s = env.reset()
        done = False

        while not done:
            a = epsilon_greedy(Q, s, epsilon)
            s2, r, done, _ = env.step(a)

            # Q-learning update:
            # Q(s,a) <- (1-α)Q(s,a) + α * [ r + γ * max_a' Q(s',a') ]
            best_next = max(Q[s2])
            td_target = r + gamma * best_next
            Q[s][a] = (1 - alpha) * Q[s][a] + alpha * td_target

            s = s2

        # decay epsilon per episode
        epsilon = max(eps_end, epsilon * eps_decay)

    return Q

# -----------------------------
# 3) Utilities: print policy + run a greedy demo
# -----------------------------
ARROWS = {0: "↑", 1: "→", 2: "↓", 3: "←"}

def greedy_action(Q, s):
    max_q = max(Q[s])
    best = [a for a, q in enumerate(Q[s]) if q == max_q]
    return random.choice(best)

def print_policy(env, Q):
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            pos = (r, c)
            if pos == env.goal:
                row.append("G")
            elif pos == env.pit:
                row.append("P")
            elif pos == env.start:
                row.append("S")
            else:
                s = env._state(pos)
                a = greedy_action(Q, s)
                row.append(ARROWS[a])
        print(" ".join(f"{x:>1}" for x in row))

def run_greedy_episode(env, Q, render=True):
    s = env.reset()
    total = 0.0
    done = False
    path = [env.pos]

    while not done:
        a = greedy_action(Q, s)
        s, r, done, _ = env.step(a)
        total += r
        path.append(env.pos)

    if render:
        print("Path:", path)
        print("Return:", total)

    return total, path



In [2]:
if __name__ == "__main__":
    random.seed(0)

    env = GridWorld(rows=5, cols=5, start=(0,0), goal=(4,4), pit=(3,3))
    Q = train_q_learning(env, episodes=8000, alpha=0.1, gamma=0.99,
                         eps_start=1.0, eps_end=0.05, eps_decay=0.999)

    print("\nLearned greedy policy (S=start, G=goal, P=pit):")
    print_policy(env, Q)

    print("\nGreedy rollout:")
    run_greedy_episode(env, Q, render=True)


Learned greedy policy (S=start, G=goal, P=pit):
S ↓ ↓ ↓ ↓
↓ ↓ ↓ ↓ ↓
↓ ↓ ↓ → ↓
→ ↓ ↓ P ↓
→ → → → G

Greedy rollout:
Path: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4)]
Return: 0.9299999999999999
